# 03 & 04 — Stream-Train-and-Discard Pipeline (Stages 3 + 4 Combined)
### Ephemeral Data Ingestion & Word2Vec Model Training

**Problem solved:** Saving raw/tokenized text for 13 years across dozens of subreddits quickly exhausts Google Drive's 15 GB free tier storage limit.

**How this unified pipeline works:**
1. For each frozen period in `config/period_definitions.csv`, checks if models already exist in `manifests/training_manifest.csv` (instant skip if done).
2. Streams & tokenizes text directly into **local ephemeral VM storage** (`/content/tmp_reddit/` or `/tmp/`, NOT Google Drive).
3. Trains Word2Vec models across all candidate seeds (e.g. 1047, 2048, 9182) from the local scratch file.
4. Saves only the lightweight `.model` files, normalized `.kv` vectors, and `.nfo.json` provenance to Google Drive / GitHub.
5. **Immediately deletes the temporary text shard**, freeing up 100% of scratch space before moving to the next period.

> **Zero Google Drive space used for raw text.** Only final trained models (~10–30 MB each) and manifests are persisted.

In [ ]:
# Cell 1 — CONFIGURATION & FILTERING
PERIOD_FILTER = None        # e.g. ['w2v__askacademia__2019q1']; None = all sufficient periods
SUB_FILTER = None           # e.g. 'AskAcademia'; None = all subreddits
SEED_FILTER = None          # e.g. [1047, 2048, 9182]; None = all config seed_candidates
MAX_PERIODS = None          # Cap on number of periods to process in this run (None = uncapped)
AUTO_GIT_PUSH = False       # True = auto-commit & push to GitHub after each period completes
DRY_RUN = False             # True = use synthetic text offline (for testing plumbing without network)

print(f"Stream-Train Pipeline: periods={PERIOD_FILTER} sub={SUB_FILTER} auto_push={AUTO_GIT_PUSH} dry={DRY_RUN}")

In [ ]:
# Cell 2 — Setup: resolve paths, load config, initialize gensim and manifests
import os, sys, csv, json, gzip, time, gc, hashlib, datetime, subprocess
from pathlib import Path
from collections import defaultdict
import yaml

from src.paths import get_project_root, resolve_tmp
from src.storage import atomic_write_bytes, atomic_write_text, sha256_file, save_gensim_atomic
from src.cleaner import clean_and_tokenize, extract_text, lang_of
from src.manifests import TRAINING_COLS, load_manifest, upsert_manifest_row
from src.api import api_get, retry_get

ROOT = get_project_root()
cfg_path = ROOT / "config/project_config.yaml"
cfg = yaml.safe_load(open(cfg_path, encoding="utf-8"))
CFG_SHA = sha256_file(cfg_path)
E = cfg["embeddings"]

try:
    from gensim.models import Word2Vec, KeyedVectors
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "gensim"])
    from gensim.models import Word2Vec, KeyedVectors

import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/03_04_stream_train__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("stream_train"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP, encoding="utf-8"); fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.INFO)
lg.addHandler(fh); lg.addHandler(sh)

TMP = resolve_tmp(ROOT, cfg)
TMAN = ROOT / "manifests/training_manifest.csv"
if not TMAN.exists():
    atomic_write_text(TMAN, ",".join(TRAINING_COLS) + "\n")

def trows(): return load_manifest(TMAN)
def tupsert(row):
    row = dict(row); row["seed"] = str(row["seed"])
    upsert_manifest_row(TMAN, row, ["model_id", "seed"], TRAINING_COLS)

print(f"Setup ready | Config {cfg['config_version']} | Ephemeral scratch tmp: {TMP}")

In [ ]:
# Cell 3 — MAIN STREAM -> TRAIN -> DISCARD LOOP
# Downloads text into Colab's local /tmp, trains all seeds, saves models to Drive/GitHub, then DELETES text.

class EphemeralSentenceStream:
    def __init__(self, path):
        self.path = Path(path)
    def __iter__(self):
        with gzip.open(self.path, "rt", encoding="utf-8") as f:
            for line in f:
                if not line or line.startswith("#"):
                    continue
                try:
                    rec = json.loads(line)
                    toks = rec.get("tokens")
                    if isinstance(toks, list) and len(toks) >= 3:
                        yield toks
                except Exception:
                    continue

def to_unix(d_str):
    return int(datetime.datetime.fromisoformat(d_str).replace(tzinfo=datetime.timezone.utc).timestamp())

def week_bounds(s, e):
    out, cur = [], datetime.datetime.fromisoformat(s)
    end = datetime.datetime.fromisoformat(e)
    while cur < end:
        nxt = min(cur + datetime.timedelta(days=7), end)
        out.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return out

PDEF_PATH = ROOT / "config/period_definitions.csv"
assert PDEF_PATH.exists(), "Missing config/period_definitions.csv — run Notebook 02 first."
pdef_rows = [r for r in csv.DictReader(open(PDEF_PATH, encoding="utf-8")) if not r["model_id"].startswith("#")]
trainable = [r for r in pdef_rows if r.get("sufficiency") in ("sufficient", "axis_grade", "marginal_merge_first")]

if PERIOD_FILTER: trainable = [r for r in trainable if r["model_id"] in PERIOD_FILTER]
if SUB_FILTER: trainable = [r for r in trainable if r["subreddit_or_group"] == SUB_FILTER]
if MAX_PERIODS: trainable = trainable[:MAX_PERIODS]

seeds_to_train = SEED_FILTER or E.get("seed_candidates", [1047, 2048, 9182])
print(f"Periods to process: {len(trainable)} | Seeds per period: {seeds_to_train}")

endpoint_map = {
    "comments": "https://arctic-shift.photon-reddit.com/api/comments/search",
    "submissions": "https://arctic-shift.photon-reddit.com/api/posts/search"
}

for pr_idx, pr in enumerate(trainable, 1):
    mid = pr["model_id"]
    sub = pr["subreddit_or_group"]
    corpus = pr.get("corpus_type", "tracked")
    span = mid.split("__")[-1]
    frac = float(pr.get("sampling_fraction", 1.0) or 1.0)
    
    # 1. Check if all seeds for this period are already trained
    existing_manifest = {(r["model_id"], r["seed"]): r for r in trows()}
    needed_seeds = []
    for s in seeds_to_train:
        stem = f"w2v__{''.join(c for c in sub.lower() if c.isalnum())}__{span}__cfg-{cfg['config_version']}__seed-{s}"
        mpath = ROOT / "models/word2vec" / corpus / sub.lower() / (stem + ".model")
        rec = existing_manifest.get((mid, str(s)), {})
        if rec.get("status") == "complete" and mpath.exists() and sha256_file(mpath) == rec.get("model_sha256", ""):
            continue
        needed_seeds.append((s, stem, mpath))
        
    if not needed_seeds:
        lg.info(f"[{pr_idx}/{len(trainable)}] SKIP: All seeds complete for {mid}")
        continue
        
    lg.info(f"[{pr_idx}/{len(trainable)}] START: {mid} ({sub}) -> training seeds: {[s for s, _, _ in needed_seeds]}")
    
    # 2. Stream data to local ephemeral scratch file
    scratch_file = TMP / f"ephemeral__{sub.lower()}__{span}__{int(time.time())}.jsonl.gz"
    scratch_tmp = scratch_file.with_suffix(scratch_file.suffix + ".tmp")
    n_rec = n_tok = 0
    seen_hashes = set()
    
    try:
        with gzip.open(scratch_tmp, "wt", encoding="utf-8") as fz:
            fz.write("#manifest " + json.dumps({"period": mid, "sub": sub, "config": cfg["config_version"]}) + "\n")
            
            if DRY_RUN:
                for i in range(400):
                    text = f"Academic research and teaching note {i} in {sub}, challenging but not impossible."
                    toks, _ = clean_and_tokenize(text)
                    if toks:
                        rec = {"sub": sub, "ts": f"{pr['start_date']}T00:00:00Z", "tokens": toks}
                        fz.write(json.dumps(rec) + "\n")
                        n_rec += 1; n_tok += len(toks)
            else:
                for ctype in ["comments", "submissions"]:
                    for ws, we in week_bounds(pr["start_date"], pr["end_date"]):
                        after_u, before_u = to_unix(ws), to_unix(we)
                        after = after_u
                        while True:
                            r = retry_get(endpoint_map[ctype], params={"subreddit": sub, "after": after, "before": before_u, "limit": 100, "sort": "asc", "fields": "id,created_utc,body,title,selftext"}, tries=4)
                            batch = r.json().get("data", [])
                            if not batch: break
                            for item in batch:
                                rid = str(item.get("id", ""))
                                if frac < 1.0 and (int(hashlib.sha256(rid.encode()).hexdigest(), 16) % 10000) >= frac * 10000:
                                    continue
                                raw = extract_text(ctype, item)
                                toks, fl = clean_and_tokenize(raw)
                                if not toks: continue
                                lang, _ = lang_of(raw, allow_heuristic=True)
                                if lang != "en": continue
                                kh = hashlib.sha256(" ".join(toks).encode()).hexdigest()[:16]
                                if kh in seen_hashes: continue
                                if len(seen_hashes) < 500000: seen_hashes.add(kh)
                                
                                rec = {"sub": sub, "tokens": toks}
                                fz.write(json.dumps(rec) + "\n")
                                n_rec += 1; n_tok += len(toks)
                            after = int(batch[-1].get("created_utc", after)) + 1
                            if len(batch) < 100: break
        
        os.replace(scratch_tmp, scratch_file)
        lg.info(f"  Ingested to scratch: {n_rec} docs (~{n_tok} tokens, {scratch_file.stat().st_size/1024:.1f} KB)")
        
        # 3. Train each required seed from the ephemeral scratch file
        stream = EphemeralSentenceStream(scratch_file)
        initial_lr = float(E["lr"]["initial"]) if isinstance(E.get("lr"), dict) else 0.025
        min_lr = float(E["lr"]["min"]) if isinstance(E.get("lr"), dict) else 0.0001
        total_epochs = int(E.get("epochs", 5))
        
        for seed_val, stem, target_mpath in needed_seeds:
            t0 = time.time()
            lg.info(f"  Training Word2Vec: {stem} (seed {seed_val})...")
            
            model = Word2Vec(vector_size=E["dim"], window=E["window"], sg=1, negative=E["negative"],
                             min_count=E["min_count"] if not DRY_RUN else 1, sample=E["subsample"],
                             workers=E.get("workers", 2), seed=int(seed_val),
                             alpha=initial_lr, min_alpha=min_lr, epochs=1)
            model.build_vocab(stream)
            
            for ep in range(total_epochs):
                ep_alpha = initial_lr - (initial_lr - min_lr) * (ep / total_epochs)
                ep_min_alpha = initial_lr - (initial_lr - min_lr) * ((ep + 1) / total_epochs)
                model.train(stream, total_examples=model.corpus_count, epochs=1, start_alpha=ep_alpha, end_alpha=ep_min_alpha)
                
            # Save model atomically
            save_gensim_atomic(model, target_mpath)
            model_sha = sha256_file(target_mpath)
            train_secs = round(time.time() - t0, 2)
            
            # Export normalized KeyedVectors
            vec_path = ROOT / "vectors" / (target_mpath.stem.replace("w2v__", "vectors_norm__") + ".kv")
            model.wv.fill_norms()
            save_gensim_atomic(model.wv, vec_path)
            vec_sha = sha256_file(vec_path)
            
            # Save sidecar provenance
            nfo = {"model_id": mid, "seed": seed_val, "spec": {"dim": E["dim"], "window": E["window"]},
                   "config_version": cfg["config_version"], "vocab_size": len(model.wv),
                   "train_secs": train_secs, "docs_trained": n_rec, "tokens_trained": n_tok}
            atomic_write_text(target_mpath.with_suffix(".nfo.json"), json.dumps(nfo, indent=2))
            
            # Update manifest
            trow = {"model_id": mid, "corpus_type": corpus, "subreddit_or_group": sub, "period_id": mid,
                    "spec_hash": "spec0", "dim": E["dim"], "window": E["window"], "sg": 1,
                    "negative": E["negative"], "epochs": total_epochs, "min_count": E["min_count"],
                    "max_vocab": E["max_vocab"], "subsample": E["subsample"], "workers": E.get("workers", 2),
                    "lr": str(initial_lr), "seed": str(seed_val), "vocab_size": len(model.wv),
                    "words_processed": model.corpus_total_words, "epochs_done": total_epochs,
                    "train_secs": train_secs, "peak_ram_mb": "n/a", "model_path": str(target_mpath),
                    "vectors_path": str(vec_path), "model_sha256": model_sha, "vectors_sha256": vec_sha,
                    "config_version": cfg["config_version"], "status": "complete", "diagnostics_path": ""}
            tupsert(trow)
            lg.info(f"    Saved model + vectors: vocab={len(model.wv)} in {train_secs}s")
            del model; gc.collect()
            
    finally:
        # 4. PURGE EPHEMERAL SCRATCH FILE (Guarantees zero persistent storage for raw text)
        if scratch_file.exists():
            scratch_file.unlink(missing_ok=True)
        if scratch_tmp.exists():
            scratch_tmp.unlink(missing_ok=True)
        lg.info(f"  [CLEANUP] Deleted scratch text shard: {scratch_file.name} (0 KB disk retained)")
        
    # 5. Optional auto Git push after each period
    if AUTO_GIT_PUSH:
        try:
            subprocess.run(["git", "add", "manifests/", "models/", "vectors/"], cwd=ROOT, check=True)
            subprocess.run(["git", "commit", "-m", f"checkpoint: completed period {mid}"], cwd=ROOT, check=True)
            subprocess.run(["git", "push", "origin", "arena/01a06e48-reddit-embeddings"], cwd=ROOT, check=True)
            lg.info(f"  [GIT] Checkpoint pushed to GitHub for {mid}")
        except Exception as ge:
            lg.warning(f"  [GIT] Auto-push deferred: {ge}")

print("=" * 70)
print("STREAM-TRAIN PIPELINE COMPLETE: All requested periods trained and scratch text purged!")
print("=" * 70)

In [ ]:
# Cell 4 — SEED EVALUATION (Stage 9 Rollup)
# Prespecified stability metric across trained seeds
SE = cfg.get("seed_eval", {})
DN, DK = int(SE.get("diag_words", 200)), int(SE.get("neighbor_k", 10))
MINJ = float(SE.get("min_cross_seed_jaccard", 0.30))

by_model = defaultdict(list)
for r in trows():
    if r.get("status") == "complete":
        by_model[r["model_id"]].append(r)

prod, seedrep = {}, []
for mid, rows in sorted(by_model.items()):
    mods = {}
    for r in rows:
        try:
            vp = Path(r.get("vectors_path", ""))
            if vp.exists(): mods[int(r["seed"])] = KeyedVectors.load(str(vp))
            else: mods[int(r["seed"])] = Word2Vec.load(r["model_path"]).wv
        except Exception as e:
            lg.error(f"Seed load error {mid}: {e}")
            
    if len(mods) < 2:
        best = sorted(mods)[0] if mods else None
        if best: prod[mid] = best
        seedrep.append({"model_id": mid, "production_seed": best, "note": "single-seed"})
        continue
        
    common = set.intersection(*[set(m.index_to_key[:5000]) for m in mods.values()])
    freq = sorted(common, key=lambda w: -min(m.get_vecattr(w, "count") if hasattr(m, "get_vecattr") else 1 for m in mods.values()))[:DN]
    nb = {s: {w: set(x for x, _ in m.most_similar(w, topn=DK)) for w in freq if w in m} for s, m in mods.items()}
    
    def jac(a, b): return len(a & b) / len(a | b) if (a | b) else 0.0
    meanj = {}
    for s in mods:
        js = [jac(nb[s][w], nb[o][w]) for o in mods if o != s for w in freq if w in nb[s] and w in nb[o]]
        meanj[s] = sum(js) / max(1, len(js))
    best = sorted(meanj, key=lambda s: (-meanj[s], s))[0]
    prod[mid] = best
    seedrep.append({"model_id": mid, "production_seed": best, "jaccard": {str(k): round(v, 3) for k, v in meanj.items()}})
    print(f"Period {mid}: Best production seed = {best} (Stability Jaccard: {meanj[best]:.3f})")
    del mods; gc.collect()

atomic_write_text(ROOT / "models/production_seed.json", json.dumps({"seeds": prod, "report": seedrep}, indent=2))
print(f"Production seeds saved -> models/production_seed.json ({len(prod)} models analyzed)")